# DPN Classification - YOLOv11 Training (Google Colab)

Trains YOLOv11 on a free T4 GPU - ~15 min vs 3+ hrs on CPU.

## Before running:
1. **Enable GPU**: Runtime -> Change runtime type -> T4 GPU -> Save
2. **Run all**: Runtime -> Run all

## After training:
- best_yolo_model.pt is saved to Google Drive under DPN_Checkpoints/
- Download it, place in checkpoints/ on your PC, restart the API.

In [ ]:
import torch
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU - go to Runtime -> Change runtime type -> T4 GPU -> Save')


In [ ]:
import sys, subprocess, shutil
from pathlib import Path

PROJECT_DIR = '/content/dpn_project'
GITHUB_REPO = 'https://github.com/catnipp9/transistors-thermal-ai-testing.git'

if Path(PROJECT_DIR).exists():
    shutil.rmtree(PROJECT_DIR)

result = subprocess.run(['git', 'clone', GITHUB_REPO, PROJECT_DIR], capture_output=True, text=True)
if result.returncode != 0:
    print('Clone FAILED:', result.stderr)
    raise RuntimeError('Could not clone repo')

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

from models.data_loader import prepare_yolo_dataset
from models.model import YOLOv11DPNClassifier
from models.trainer import YOLOTrainer
print('Repo cloned and modules loaded.')

log = subprocess.run(['git', 'log', '--oneline', '-3'], capture_output=True, text=True, cwd=PROJECT_DIR)
print('Latest commits:')
print(log.stdout)


In [ ]:
!pip install ultralytics>=8.3.0 scikit-learn>=1.4.0 scipy>=1.12.0 pyyaml>=6.0.0 gdown --quiet
print('Packages installed.')


In [ ]:
# Mount Google Drive and locate the dataset
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

# Dataset is in: Shared with me > SISON-THESIS > ORIGINAL > ThermoDataBase
# Shared folders appear under /content/drive/MyDrive/... when added to your Drive
# or under /content/drive/Shareddrives/...
# We search all possible locations automatically

def find_folder(name, search_roots):
    """Recursively search for a folder by name under given roots."""
    for root in search_roots:
        root_path = Path(root)
        if not root_path.exists():
            continue
        # Check up to 4 levels deep
        for path in root_path.rglob(name):
            if path.is_dir():
                return str(path)
    return None

search_roots = [
    '/content/drive/MyDrive',
    '/content/drive/Shareddrives',
    '/content/drive/Shared drives',
    '/content/drive',
]

DRIVE_DATASET = find_folder('ThermoDataBase', search_roots)

if DRIVE_DATASET:
    print(f'Found dataset at: {DRIVE_DATASET}')
else:
    # Print what's available to help debug
    print('ThermoDataBase not found automatically.')
    print('Available top-level Drive contents:')
    for root in search_roots:
        root_path = Path(root)
        if root_path.exists():
            print(f'\n{root}/')
            for item in sorted(root_path.iterdir()):
                print(f'  {item.name}/')
    raise RuntimeError('Could not find ThermoDataBase. See folder list above.')

# Verify contents
control_dir = Path(DRIVE_DATASET) / 'Control Group'
dm_dir      = Path(DRIVE_DATASET) / 'DM Group'
control_count = len([d for d in control_dir.iterdir() if d.is_dir()]) if control_dir.exists() else 0
dm_count      = len([d for d in dm_dir.iterdir() if d.is_dir()]) if dm_dir.exists() else 0

print(f'Control Group : {control_count} subjects')
print(f'DM Group      : {dm_count} subjects')

if control_count == 0 and dm_count == 0:
    print('\nContents of ThermoDataBase:')
    for item in sorted(Path(DRIVE_DATASET).iterdir()):
        print(f'  {item.name}')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from pathlib import Path

CHECKPOINT_DIR = '/content/drive/MyDrive/DPN_Checkpoints'
YOLO_DATASET   = '/content/yolo_dataset'

Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)

print(f'Dataset       : {DRIVE_DATASET}')
print(f'Checkpoints   : {CHECKPOINT_DIR}')
print(f'PyTorch       : {torch.__version__}')
print(f'CUDA          : {torch.cuda.is_available()}')

In [ ]:
# Build YOLO dataset with oversampling to fix class imbalance
yaml_path = prepare_yolo_dataset(
    data_dir=DRIVE_DATASET,
    output_dir=YOLO_DATASET,
)
print(f'Dataset ready: {yaml_path}')


## Training Settings

| Setting | Value | Notes |
|---------|-------|-------|
| `YOLO_VARIANT` | `yolo11l-cls` | Large model - best for GPU |
| `EPOCHS` | `100` | More = better accuracy |
| `PATIENCE` | `20` | Early stopping patience |
| `BATCH` | `32` | Larger batch works well on GPU |

In [ ]:
# Edit these settings to improve accuracy
YOLO_VARIANT = 'yolo11l-cls'
EPOCHS       = 100
PATIENCE     = 20
IMGSZ        = 224
BATCH        = 32
LR           = 0.001

device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print(f'Variant : {YOLO_VARIANT}')
print(f'Epochs  : {EPOCHS} (patience={PATIENCE})')
print(f'Device  : {device_name}')

yolo_model   = YOLOv11DPNClassifier(variant=YOLO_VARIANT)
yolo_trainer = YOLOTrainer(yolo_model, save_dir=CHECKPOINT_DIR)

yolo_history = yolo_trainer.train(
    data_yaml=yaml_path,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    lr0=LR,
    patience=PATIENCE,
)
print('Training complete!')
print(f'Results: {yolo_history}')


In [ ]:
# Save best weights to Google Drive
yolo_trainer.save_best_checkpoint()

saved_path = Path(CHECKPOINT_DIR) / 'best_yolo_model.pt'
if saved_path.exists():
    size_mb = saved_path.stat().st_size / 1e6
    print(f'Model saved: {saved_path} ({size_mb:.1f} MB)')
    print('Find it in MyDrive/DPN_Checkpoints/ on your Google Drive')
else:
    print('WARNING: model file not found - check training completed successfully')


In [ ]:
# Download model directly to your PC
from google.colab import files
model_path = Path(CHECKPOINT_DIR) / 'best_yolo_model.pt'
if model_path.exists():
    files.download(str(model_path))
    print('Download started. Place the file in checkpoints/ on your PC.')
    print('Then restart the API: uvicorn api.main:app --reload')
else:
    print('Model not found - make sure training completed.')


## Train sklearn Model (Temperature CSV — Required for Combined Endpoint)

The `/predict/patient/combined` endpoint fuses **YOLOv11 (images) + sklearn (CSVs)** per foot.
This cell trains the sklearn model on raw temperature matrices and saves it to Drive.

**Feature design** — 54 angiosome-aligned features per foot (Hernandez-Contreras 2019):
- MPA (Medial Plantar Artery): top 60% rows × inner 35% cols
- LPA (Lateral Plantar Artery): top 60% rows × outer 65% cols  ← **top discriminator**
- MCA (Medial Calcaneal Artery): bottom 40% rows × inner 35% cols
- LCA (Lateral Calcaneal Artery): bottom 40% rows × outer 65% cols  ← **2nd discriminator**
- Plus: 12 global stats, 6 inter-angiosome diffs, 3 gradients, 6 hot/cold spots, 3 forefoot/hindfoot

Best model on this dataset: **SVM with RBF kernel** (~89% test accuracy).
It acts as the secondary signal — YOLO contributes 60% weight, sklearn 40%.

In [ ]:
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.svm import SVC
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
)
from sklearn.metrics import accuracy_score, classification_report
from scipy.ndimage import zoom

# Import the same feature extractor used by the API at inference time
import sys
sys.path.insert(0, PROJECT_DIR)
from models.preprocessing import extract_thermal_features

TARGET_H, TARGET_W = 168, 65

# --- Load CSV temperature matrices and extract features ---
print("Loading CSVs and extracting angiosome-aligned features...")
X, y = [], []

for group_name, label in [('Control Group', 0), ('DM Group', 1)]:
    group_dir = Path(DRIVE_DATASET) / group_name
    if not group_dir.exists():
        print(f'Warning: {group_dir} not found')
        continue
    for subject_folder in sorted(group_dir.iterdir()):
        if not subject_folder.is_dir():
            continue
        subject_id = subject_folder.name
        for side in ['L', 'R']:
            csv_path = subject_folder / f'{subject_id}_{side}.csv'
            if csv_path.exists():
                try:
                    data = pd.read_csv(csv_path, header=None).values.astype(np.float32)
                    if data.shape != (TARGET_H, TARGET_W):
                        zf = (TARGET_H / data.shape[0], TARGET_W / data.shape[1])
                        data = zoom(data, zf, order=1)
                    X.append(extract_thermal_features(data))
                    y.append(label)
                except Exception as e:
                    print(f'Skipping {csv_path.name}: {e}')

X = np.array(X)
y = np.array(y)
print(f'Loaded {len(X)} samples  |  Control: {(y==0).sum()}, Diabetic: {(y==1).sum()}')
print(f'Features per sample: {X.shape[1]} (54 angiosome-aligned features)')

# --- Train/test split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# --- Model comparison ---
candidates = {
    'SVM (RBF)': Pipeline([('scaler', StandardScaler()),
        ('clf', SVC(kernel='rbf', C=1.0, gamma='scale', class_weight='balanced', probability=True, random_state=42))]),
    'Gradient Boosting': Pipeline([('scaler', StandardScaler()),
        ('clf', GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42))]),
    'Random Forest': Pipeline([('scaler', StandardScaler()),
        ('clf', RandomForestClassifier(n_estimators=300, max_depth=12, class_weight='balanced', random_state=42, n_jobs=-1))]),
}

print('\n-- Model comparison (5-fold CV) --')
results = {}
for name, model in candidates.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy')
    results[name] = scores.mean()
    print(f'  {name:<22} CV acc: {scores.mean():.4f} +/- {scores.std():.4f}')

best_name = max(results, key=results.get)
print(f'\nBest: {best_name}  ({results[best_name]:.4f})')

# --- Grid search tuning ---
if best_name == 'SVM (RBF)':
    param_grid = {'clf__C': [0.1, 1.0, 10.0, 100.0], 'clf__gamma': ['scale', 'auto', 0.001, 0.01]}
elif best_name == 'Gradient Boosting':
    param_grid = {'clf__n_estimators': [100, 200, 300], 'clf__learning_rate': [0.05, 0.1, 0.2], 'clf__max_depth': [3, 4, 5]}
else:
    param_grid = {'clf__n_estimators': [200, 300, 500], 'clf__max_depth': [8, 12, None]}

gs = GridSearchCV(candidates[best_name], param_grid, cv=cv, scoring='accuracy', n_jobs=-1, verbose=1)
gs.fit(X_train, y_train)
best_model = gs.best_estimator_
print(f'Best params : {gs.best_params_}')
print(f'Best CV acc : {gs.best_score_:.4f}')

# --- Test set evaluation ---
y_pred = best_model.predict(X_test)
print(f'\nTest accuracy: {accuracy_score(y_test, y_pred):.4f}')
print(classification_report(y_test, y_pred, target_names=['Control', 'Diabetic']))

# --- Save to Drive ---
best_model.fit(X_train, y_train)  # retrain on full train set
sklearn_path = Path(CHECKPOINT_DIR) / 'best_sklearn_model.joblib'
joblib.dump(best_model, sklearn_path)
print(f'\nSaved: {sklearn_path}')

In [ ]:
# Download sklearn model directly to your PC
from google.colab import files
sklearn_path = Path(CHECKPOINT_DIR) / 'best_sklearn_model.joblib'
if sklearn_path.exists():
    files.download(str(sklearn_path))
    print('Download started. Place the file in checkpoints/ on your PC.')
    print('Both models are now ready — restart the API and use /predict/patient/combined.')
else:
    print('sklearn model not found — check the training cell ran successfully.')
